# ORCHESTRA-WM · Phase 4E on Google Colab

**Choose Runtime → Run all, then authorize Google Drive when prompted.** The default settings run the frozen campaign and resume previous progress automatically. Use a CPU runtime; this protocol deliberately uses two CPU threads.

This notebook restores your verified **November 21 and 22** datasets, processes **November 23 and 25** directly from your mounted original ZIPs, backs up completed data, and trains the paired models. Independent November 28/29 forecasts run **only if validation and readiness gates pass**. No OAuth secrets or Codex environment publishing are needed.

Your original ZIP folder and the existing completed-date backup manifests/parts must be visible in **My Drive**. For a folder under **Shared with me**, add a shortcut to My Drive once. All new data, checkpoints and backups stay in `MyDrive/ORCHESTRA_WM_PHASE4E_COLAB`.

**Disconnected?** Reopen this same notebook and Run all with the same folder. Do not run two sessions against that folder concurrently. Initial setup takes a few minutes; the research campaign can span multiple Colab sessions. No scientific results are claimed until your run produces them.

In [ ]:
#@title 1 · Mount Drive and choose where to run
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
PRIVATE_ROOT = DRIVE_ROOT / 'ORCHESTRA_WM_PHASE4E_COLAB'
SOURCE_ROOT = "" #@param {type:"string"}
RUN_STAGE = "all" #@param ["all", "prepare", "train-validation", "independent-test"]

# Leave SOURCE_ROOT blank to discover the originals automatically.
# If archive names are duplicated, enter their intended mounted folder path.
# The default "all" performs preparation, training and strictly gated testing.
assert DRIVE_ROOT.is_dir(), 'Google Drive is not mounted.'
assert not SOURCE_ROOT or Path(SOURCE_ROOT).is_dir(), 'SOURCE_ROOT must be a mounted directory.'
print('Persistent workspace:', PRIVATE_ROOT)

In [ ]:
#@title 2 · Install the pinned implementation and frozen dependencies
import subprocess
import sys

CODE_REVISION = "72286edda0ca9a3bb620f9251bac0522869e1710"
REPO_URL = "https://github.com/jsp2195/orchestra-wm.git"
REPO = Path('/content/orchestra-wm-colab')

def command(args, cwd=None):
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)

if not (REPO / '.git').exists():
    command(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, REPO])
else:
    origin = subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip()
    assert origin == REPO_URL, 'Existing checkout has a different origin; preserve it and inspect.'
    command(['git', 'diff', '--quiet'], REPO)
    command(['git', 'diff', '--cached', '--quiet'], REPO)
command(['git', 'fetch', '--depth=1', 'origin', CODE_REVISION], REPO)
command(['git', 'checkout', '--detach', CODE_REVISION], REPO)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == CODE_REVISION
command([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
command([sys.executable, '-m', 'uv', 'sync', '--frozen', '--python', '3.11'], REPO)
PYTHON = REPO / '.venv/bin/python'
command([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
         '-r', 'configs/i24_phase4e_requirements.txt'], REPO)
print('Pinned code and locked CPU dependencies are ready.')

In [ ]:
#@title 3 · Verify recovery, interruption handling and scientific gates
command([PYTHON, '-m', 'pytest', 'tests/i24_phase4e_colab', 'tests/i24_phase4e',
         '--import-mode=importlib', '-q'], REPO)
print('Preflight passed. These are software fixtures, not research outcomes.')

### Run and resume
The next cell does the work. Progress, checkpoints and compressed verified backups are written to Drive automatically. November 21/22 are restored and reused; they are never silently reingested.

An interrupted ZIP is replayed up to its committed record checkpoint, then conversion resumes. Training saves both paired models, optimizer and RNG state after each update. Failed promotion or an unresolved training-stability check leaves independent forecasts unopened; it does not change the frozen budget or tune on tests.

Mounted Drive backups are closed, reopened and SHA256-verified. The notebook performs no sharing operation. Keep the newly created My Drive folder private; it does not independently query Drive sharing permissions.

In [ ]:
#@title 4 · Process the data, train, validate, and run only gated tests
import signal

args = [str(PYTHON), '-u', str(REPO / 'scripts/run_i24_phase4e_colab.py'),
        '--drive-root', str(DRIVE_ROOT), '--private-root', str(PRIVATE_ROOT), '--stage', RUN_STAGE]
if SOURCE_ROOT:
    args += ['--source-root', SOURCE_ROOT]
process = subprocess.Popen(args, cwd=REPO, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.terminate()
    print('Stopped. Keep the same notebook and Drive folder to resume saved progress.')
    raise
if returncode:
    raise RuntimeError('The run stopped safely. Read the last error above; saved data/checkpoints remain in Drive. '
                       'For a disconnect, rerun this notebook. For missing files, make their folders visible in My Drive.')

In [ ]:
#@title 5 · View actual saved status and output locations
import json
from IPython.display import display, Markdown

workspace = PRIVATE_ROOT / 'workspace'
status_path = workspace / 'work/phase4e/colab_execution.json'
if status_path.exists():
    display(Markdown('**Saved run status**'))
    print(json.dumps(json.loads(status_path.read_text()), indent=2))

output = workspace / 'outputs/i24_phase4e'
for date in ['2022-11-21', '2022-11-22', '2022-11-23', '2022-11-25', '2022-11-28', '2022-11-29']:
    path = output / 'data_v4' / date / 'data_manifest.json'
    if path.exists():
        print(date, 'completed windows:', json.loads(path.read_text())['scene_counts'])
for seed in [101, 202]:
    path = output / f'residual_seed{seed}' / 'completed.json'
    if path.exists():
        result = json.loads(path.read_text())
        print('Seed', seed, 'updates/model:', result['updates_per_model'], 'selected update:', result['selected_update'])
print('Results:', output)
print('Verified backup archives and receipts:', PRIVATE_ROOT / 'backups')
print('Two independent test dates cannot satisfy the five-independent-group confirmation gate.')

### Files to inspect after your run

- `source_qc_colab_train_validation/`: four-date source quality and split checks.
- `residual_seed101/` and `residual_seed202/`: matched models, learning curves and completion receipts.
- `validation/` and `calibration64_seed101/`, `calibration64_seed202/`: primary comparisons and separate calibration diagnostics.
- `colab_readiness.json`: promotion and training-readiness decision, including any reason tests remain unopened.
- `independent_test/`: produced only after the test gate; per-day/regime scores, paired day/seed effects and equal-day summary.

[Workflow and recovery details](https://github.com/jsp2195/orchestra-wm/blob/phase4-i24-continuous/docs/I24_PHASE4E_COLAB.md) · [Frozen scientific protocol](https://github.com/jsp2195/orchestra-wm/blob/phase4-i24-continuous/docs/I24_PHASE4E_PROTOCOL.md)

Do not change seeds, dates, horizons, selection rules or model architecture after opening test outcomes. Macro forecast improvements do not establish microscopic interaction benefit or a complete joint traffic world model.